# Final Project Phase 2 Summary
This Jupyter Notebook (.ipynb) will serve as the skeleton file for your submission for Phase 2 of the Final Project. Answer all statements addressed below as specified in the instructions for the project, covering all necessary details. Please be clear and concise in your answers. Each response should be at most 3 sentences. Good luck! <br><br>

Note: To edit a Markdown cell, double-click on its text.

## Jupyter Notebook Quick Tips
Here are some quick formatting tips to get you started with Jupyter Notebooks. This is by no means exhaustive, and there are plenty of articles to highlight other things that can be done. We recommend using HTML syntax for Markdown but there is also Markdown syntax that is more streamlined and might be preferable. 
<a href = "https://towardsdatascience.com/markdown-cells-jupyter-notebook-d3bea8416671">Here's an article</a> that goes into more detail. (Double-click on cell to see syntax)

# Heading 1
## Heading 2
### Heading 3
#### Heading 4
<br>
<b>BoldText</b> or <i>ItalicText</i>
<br> <br>
Math Formulas: $x^2 + y^2 = 1$
<br> <br>
Line Breaks are done using br enclosed in < >.
<br><br>
Hyperlinks are done with: <a> https://www.google.com </a> or 
<a href="http://www.google.com">Google</a><br>

# Data Collection and Cleaning
You are required to provide data collection and cleaning for the three (3) minimum datasets. Create a function for each of the following sections that reads or scrapes data from a file or website, manipulate and cleans the parsed data, and writes the cleaned data into a new file. 

Make sure your data cleaning and manipulation process is not too simple. Performing complex manipulation and using modules not taught in class shows effort, which will increase the chance of receiving full credit.


## Data Sources
Include sources (as links) to your datasets. Add any additional data sources if needed. Clearly indicate if a data source is different from one submitted in your Phase I, as we will check that it satisfies the requirements.
*   Downloaded Dataset Source: https://www.ers.usda.gov/data-products/season-average-price-forecasts. 
*   Web Collection #1 Source: https://www.ncdc.noaa.gov/stormevents/  
*   Web Collection #2 Source: https://weatherstack.com/documentation. (Historical Weather)



## Downloaded Dataset Requirement

Fill in the predefined functions with your data scraping/parsing code. You may modify/rename each function as you seem fit, but you must provide at least 3 separate functions that clean each of your required datasets.


In [3]:
import pandas as pd
import numpy as np

def data_parser(file_name:str) -> pd.DataFrame:
    df = pd.read_csv(file_name, index_col=0)

    # Most of exchange are NaN because the item aren't supposed to be in the market for options such as Farm Bill direct Payment rate, etc.
    # Fill futures_exchange
    
    df.loc["Corn", "futures_exchange"] = df.loc["Corn", "futures_exchange"].fillna("None")
    df.loc["Cotton", "futures_exchange"] = df.loc["Cotton", "futures_exchange"].fillna("None")
    df.loc["Soybeans", "futures_exchange"] = df.loc["Soybeans", "futures_exchange"].fillna("None")
    # For Wheat these type of wheat production are mostly traded at CBOT
    df.loc["Wheat", "futures_exchange"] = df.loc["Wheat", "futures_exchange"].fillna("CBOT")
    
    # Drop commodity_class, only Wheat production by class (item) has this attributes since others item aren't defined
    # and we can't possibly work with these since 90%+ data aren't defined

    df.drop(["commodity_class"], axis=1, inplace=True)

    # Fill marketing_year, only Futures prices daily don't have this attributes (since it can't). Also change the datatype from float64 
    # to str so all the data are compatible and remove .0 from all the str.

    df["marketing_year"] = df["marketing_year"].astype("str")
    df.loc[:, "marketing_year"] = df.loc[:, "marketing_year"].str.replace(".0", "")
    df.loc[:, "marketing_year"] = df.loc[:, "marketing_year"].fillna("None")

    # Fill calendar_year_month & futures_contract (mostly Farm bill payment rate and wheat production)
    df.loc[:, "calendar_year_month"] = df.loc[:, "calendar_year_month"].fillna("None")
    df.loc[:, "futures_contract"] = df.loc[:, "futures_contract"].fillna("None")

    # Fill data_source_date
    df.loc[:, "data_source_date"] = df.loc[:, "data_source_date"].fillna("None")

    # Fill unit for wheat contract weight ~ normally is 5000 bushels
    df["unit"] = df["unit"].fillna("5000 bushels")
    
    # Export Data
    df.to_csv("../data/processed/usda_price_forecasts_clean.csv")
    
    return df

############ Function Call ############
df = data_parser("../data/raw/usda_price_forecasts.csv")
df
# mask = df[df.index == "Wheat"]["futures_exchange"].isna()
# df[df.index == "Wheat"][mask]["item"]
# df.loc["Wheat", :]["futures_exchange"].isna()
# df[(df.index == "Wheat") & (df["commodity_class"].str.startswith("Winter"))]

,item,futures_exchange,data_period,marketing_year,calendar_year_month,futures_contract,data_source_date,value,unit
commodity,,,,,,,,,
Corn,Basis,CBOT,Monthly,1997,1997-09,1997-12,1998-11-30,-0.145625,U.S. dollars per bushel
Corn,Basis,CBOT,Monthly,1997,1997-10,1997-12,1998-11-30,-0.251500,U.S. dollars per bushel
Corn,Basis,CBOT,Monthly,1997,1997-11,1997-12,1998-11-30,-0.241875,U.S. dollars per bushel
Corn,Basis,CBOT,Monthly,1997,1997-12,1998-03,1998-11-30,-0.205625,U.S. dollars per bushel
Corn,Basis,CBOT,Monthly,1997,1998-01,1998-03,1998-11-30,-0.144500,U.S. dollars per bushel
...,...,...,...,...,...,...,...,...,...
Wheat,Wheat production by class,CBOT,Yearly,2024,None,None,2024-06-12,208612.000000,1000 bushels
Wheat,Wheat production by class,CBOT,Yearly,2024,None,None,2024-07-12,215478.000000,1000 bushels
Wheat,Wheat production by class,CBOT,Yearly,2024,None,None,2024-08-12,223805.000000,1000 bushels


## Web Collection Requirement \#1


In [9]:
import requests
from bs4 import BeautifulSoup

def web_parser1(link):
    response = requests.get(link)
    soup = BeautifulSoup(response.text, parser="html.parser")
    rows = soup.find_all("tr")

    # event report start at row 9 and so on, the last row is just total so we don't include it
    data = rows[9:-1]
    all_data = []
    dates = []
  
    for row in data:
        td = row.find_all("td")
        place = td[0].text.lower().title()
        event = td[6].text
        date = td[3].text
        magni = "".join(td[7].text.split(" "))

        split_date = date.split("/") #Month/Day/Year
        split_date[0], split_date[1] = split_date[1], split_date[0]
        clean_date = "/".join(split_date[::-1]) #Year/Month/Day
        
        all_data.append({
            "Place": place,
            "Event": event,
            "Magnitude": magni if magni else "None"
        })
        dates.append(clean_date)
        
    df = pd.DataFrame(all_data, index = dates)

    # Clean it 

    df.index = pd.to_datetime(df.index) # Change to datetime object for better time series analysis

    # Export
    df.to_csv("../data/processed/noaa_storm_events.csv")
    
    return df
############ Function Call ############
df = web_parser1("https://www.ncdc.noaa.gov/stormevents/listevents.jsp?eventType=ALL&beginDate_mm=02&beginDate_dd=01&beginDate_yyyy=2024&endDate_mm=02&endDate_dd=28&endDate_yyyy=2025&county=COOK%3A31&hailfilter=0.00&tornfilter=0&windfilter=000&sort=DT&submitbutton=Search&statefips=17%2CILLINOIS")
df

,Place,Event,Magnitude
2024-02-08,Norwood Park,Hail,1.00in.
2024-02-27,Streamwood,Tornado,EF0
2024-02-27,Elgin,Hail,1.00in.
2024-02-27,Hoffman Estates,Tornado,EF1
2024-02-27,South Barrington,Tornado,EF1
...,...,...,...
2025-02-08,Southern Cook County (Zo...,Winter Weather,None
2025-02-08,Central Cook County (Zon...,Winter Weather,None
2025-02-12,Northern Cook County (Zo...,Winter Weather,None
2025-02-18,Northern Cook County (Zo...,Cold/wind Chill,None


## Web Collection Requirement \#2

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone


API_KEY = "API_KEY"
# Top U.S. Corn-Producing Regions 
LOCATIONS = ["Des Moines, Iowa","Cedar Rapids, Iowa","Chicago, Illinois","Peoria, Illinois","Lincoln, Nebraska",
             "Grand Island, Nebraska","Kansas City, Missouri","Minneapolis, Minnesota","Sioux Falls, South Dakota",]

def web_parser2(location):
    weather_data = []
    url = "http://api.weatherstack.com/current"
    params = {
        "access_key":API_KEY,
        "query": location
    }
    response = requests.get(url, params=params)
    data = response.json()

    for loc in location:
        params = {"access_key":API_KEY,"query": location}
        response = requests.get(url, params=params)
        data = response.json()
        
        if "current" in data:
            weather_data.append({
                "location": location,
                "date": datetime.now(timezone.utc).date(),
                "temperature": data["current"].get("temperature"),
                "humidity": data["current"].get("humidity"),
                "wind_speed": data["current"].get("wind_speed"),
                "uv_index": data["current"].get("uv_index")
            })
        time.sleep(1) 
    return weather_data

############ Function Call ############
data = web_parser2(LOCATIONS)
df = pd.DataFrame(data)
df.to_csv("../data/processed/weatherstack_corn_belt.csv")
df

,location,date,temperature,humidity,wind_speed,uv_index
0,"Des Moines, Iowa",2025-07-05,24,91,22,7
1,"Cedar Rapids, Iowa",2025-07-05,29,72,26,8
2,"Chicago, Illinois",2025-07-05,32,42,21,9
3,"Peoria, Illinois",2025-07-05,32,52,25,10
4,"Lincoln, Nebraska",2025-07-05,29,65,13,10
5,"Grand Island, Nebraska",2025-07-05,28,59,9,10
6,"Kansas City, Missouri",2025-07-05,30,66,22,7
7,"Minneapolis, Minnesota",2025-07-05,27,76,12,6
8,"Sioux Falls, South Dakota",2025-07-05,27,74,8,10


## Additional Dataset Parsing/Cleaning Functions

Write any supplemental (optional) functions here. https://fred.stlouisfed.org/series/PMAIZMTUSDM. 

In [449]:
import pandas as pd
import numpy as np

def extra_source1(link):
    df = pd.read_csv(link, index_col = 0)
    
    # Rename the columns
    df.rename(columns = {"PMAIZMTUSDM" : "Price"}, inplace = True)
    
    # Compute percent change (Daily return)
    df["pct_change"] = df["Price"].pct_change() * 100 
    
    # Log return
    df["log_return"] = np.log( df["Price"] / df["Price"].shift(1) )

    # Log difference transformation (Stationary purpose)
    df["price_log"] = np.log(df["Price"])
    df["price_log_diff"] = df["price_log"].diff()

    # Rolling average
    df["rolling_avg"] = df["Price"].rolling(30).mean()

    # Rolling standard deviation
    df["rolling_std"] = df["Price"].rolling(30).std()

    df.dropna(inplace = True) # We have more than enough data for visualization so droping some first price movement is fine

    # Change to date time object -> Easier to work with for time series analysis
    df.index = pd.to_datetime(df.index)

    # Ready for visualization

    df.to_csv("../data/processed/fred_corn_price_derived.csv")
    
    return df
    
############ Function Call ############
df = extra_source1("../data/raw/fred_corn_price.csv")
df

,Price,pct_change,log_return,price_log,price_log_diff,rolling_avg,rolling_std
observation_date,,,,,,,
1992-06-01,110.201927,0.937935,0.009336,4.702314,0.009336,108.988547,5.322022
1992-07-01,102.394058,-7.085057,-0.073486,4.628829,-0.073486,108.871665,5.429597
1992-08-01,97.539932,-4.740632,-0.048567,4.580262,-0.048567,108.579857,5.795793
1992-09-01,98.216675,0.693811,0.006914,4.587176,0.006914,108.192502,6.089726
1992-10-01,94.559769,-3.723305,-0.037944,4.549232,-0.037944,107.394550,6.258780
...,...,...,...,...,...,...,...
2025-01-01,214.359927,5.684548,0.055289,5.367657,0.055289,236.352049,51.632904
2025-02-01,221.254544,3.216374,0.031657,5.399314,0.031657,233.929508,50.531420
2025-03-01,207.751691,-6.102859,-0.062970,5.336344,-0.062970,230.436360,48.490295


In [13]:
# Define further extra source functions as necessary

#Inconsistencies
For each inconsistency (NaN, null, duplicate values, empty strings, etc.) you discover in your datasets, write at least 2 sentences stating the significance, how you identified it, and how you handled it.

1. Nan values in columns like futures_exchangem unit, item (downloaded dataset): these missing values can break the analysis and aggregations. I identified them using df.isna().sum() and handled them by filling with specific values such as "None" (for most of the columns in the downloaded dataset) and "CBOT" for the column "future_exchanges" for Wheat.

2. Inconsistent date formats (Month/Day/Year vs. Year-Month-Day) (webparser1): date mismatches can prevent a consistent time-based analysis. I standardized all dates to YYYY-MM-DD and converted them to datetime objects using pd.to_datetime().

3. Blank strings in the magnitude column (webparser1): blank strings were acting like hidden NaNs, which affects calculations. I replaced empty strings with "None" as a string instead.

4. float64 year columns with .0 suffix (downloaded dataset): year values were mistakenly stored as floats, which caused formatting issues. I converted them to strings and removed the trailing .0 using .replace().